# Práctica: Chat de preguntas frecuentes con PLN · PitLineMotor

En este cuaderno construimos un asistente sencillo que responde a las dudas de los clientes de un taller mecánico. El cliente escribe su pregunta con sus propias palabras y el asistente busca, entre las preguntas que conoce, la que más se le parece.

---

## Objetivos
1. **Preprocesamiento:** pasar a minúsculas, quitar signos, tokenizar, eliminar *stop words* y reducir cada palabra a su raíz (*stemming*).
2. **Representación del texto:** convertir las preguntas en vectores con `CountVectorizer` y `TfidfVectorizer`.
3. **Búsqueda por similitud:** comparar la pregunta del cliente con las conocidas mediante la **similitud del coseno**.
4. **Evaluación y prueba:** medir cuántas preguntas nuevas acierta y probarlo como un chat.

---

## 1. Librerías y recursos
Usamos **NLTK** para el tratamiento del texto y **scikit-learn** para la vectorización y la medida de similitud.

In [1]:
import nltk
import re
import pandas as pd
import numpy as np

# Recursos de NLTK (solo se descargan la primera vez)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Librerías cargadas.")

Librerías cargadas.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


---

## 2. Base de conocimiento del asistente
En lugar de reseñas con una etiqueta, aquí tenemos **preguntas frecuentes agrupadas por tema**. Cada tema tiene varias formas de preguntar lo mismo y una única respuesta.

In [2]:
# Varias formas de preguntar lo mismo, agrupadas por tema
preguntas = {
    'Pedir cita': [
        "¿Cómo pido cita?",
        "Quiero reservar una cita para mi coche",
        "¿Puedo pedir hora para llevar el vehículo al taller?"],
    'Estado de la reparación': [
        "¿Cómo va la reparación de mi coche?",
        "¿En qué fase está mi vehículo?",
        "Quiero saber si ya han arreglado mi coche"],
    'Factura': [
        "¿Dónde descargo la factura?",
        "Necesito la factura de la reparación",
        "¿Cómo consigo el PDF de la factura?"],
    'Pendiente de pieza': [
        "¿Por qué mi reparación está parada?",
        "¿Qué significa pendiente de pieza?",
        "Mi coche está esperando una pieza"],
    'Recoger el coche': [
        "¿Cuándo puedo recoger mi coche?",
        "¿Ya puedo ir a por el vehículo?",
        "Quiero recoger el coche del taller"],
    'Precio': [
        "¿Cuánto cuesta la mano de obra?",
        "¿Qué precio tiene la hora de trabajo?",
        "¿Cómo se calcula el precio de la reparación?"],
    'Cambiar email': [
        "¿Cómo cambio mi email?",
        "Quiero modificar mi correo electrónico",
        "Actualizar el correo de mi cuenta"],
    'Historial': [
        "¿Puedo ver reparaciones anteriores?",
        "Historial de reparaciones de mi coche",
        "¿Qué le hicieron a mi coche la última vez?"],
}

# Una respuesta por tema
respuestas = {
    'Pedir cita': "Entra en «Mis citas» y crea una cita nueva con tu vehículo, la fecha y el motivo. Quedará pendiente hasta que Recepción la confirme.",
    'Estado de la reparación': "En «Mis reparaciones» puedes ver en qué punto está cada orden: diagnosis, reparación, pendiente de pieza, prueba o finalizada.",
    'Factura': "Cuando la reparación está finalizada, entra en «Mis reparaciones» y descarga la factura en PDF.",
    'Pendiente de pieza': "Falta una pieza para continuar. El taller ya la ha pedido y, cuando llegue, el mecánico reanudará la reparación.",
    'Recoger el coche': "Cuando la reparación aparece como finalizada, el coche está listo. Pasa por Recepción, que registrará la entrega.",
    'Precio': "Se cobra el tiempo de diagnosis y reparación al precio por hora del taller, más las piezas utilizadas y el IVA. Todo aparece desglosado en la factura.",
    'Cambiar email': "Puedes cambiar tu email desde «Mi perfil».",
    'Historial': "En «Mis reparaciones» aparecen también las reparaciones anteriores con todos sus datos.",
}

# Lo pasamos a una tabla: una fila por pregunta
df = pd.DataFrame([(p, tema) for tema, lista in preguntas.items() for p in lista],
                  columns=['pregunta', 'tema'])
print("Preguntas:", len(df), "· Temas:", df['tema'].nunique())
df

Preguntas: 24 · Temas: 8


,pregunta,tema
0,¿Cómo pido cita?,Pedir cita
1,Quiero reservar una cita para mi coche,Pedir cita
2,¿Puedo pedir hora para llevar el vehículo al t...,Pedir cita
3,¿Cómo va la reparación de mi coche?,Estado de la reparación
4,¿En qué fase está mi vehículo?,Estado de la reparación
5,Quiero saber si ya han arreglado mi coche,Estado de la reparación
6,¿Dónde descargo la factura?,Factura
7,Necesito la factura de la reparación,Factura
8,¿Cómo consigo el PDF de la factura?,Factura
9,¿Por qué mi reparación está parada?,Pendiente de pieza


---

## 3. Preprocesamiento del texto

Limpiamos cada pregunta para quedarnos solo con las palabras que aportan significado:

1. **Minúsculas:** *Factura* y *factura* cuentan como la misma palabra.
2. **Quitar signos y números:** fuera ¿ ? , . y similares.
3. **Tokenización:** separar la frase en palabras.
4. **Stop words:** quitamos palabras vacías (*el*, *de*, *mi*…), **pero conservamos el "no"**, porque cambia el sentido de la frase.
5. **Stemming:** reducimos cada palabra a su raíz con `SnowballStemmer`, que sí funciona en español: *reparación* y *reparaciones* → *repar*.

In [3]:
stemmer = SnowballStemmer('spanish')
stop_words_es = set(stopwords.words('spanish')) - {'no'}

def preprocesar(texto):
    # 1. Minúsculas
    texto = texto.lower()

    # 2. Quitar signos de puntuación y números
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)

    # 3. Tokenización
    tokens = word_tokenize(texto, language='spanish')

    # 4. Quitar stop words y 5. reducir a la raíz
    raices = [stemmer.stem(t) for t in tokens if t not in stop_words_es]

    return " ".join(raices)

df['pregunta_limpia'] = df['pregunta'].apply(preprocesar)
df[['pregunta', 'pregunta_limpia']]

,pregunta,pregunta_limpia
0,¿Cómo pido cita?,com pid cit
1,Quiero reservar una cita para mi coche,quier reserv cit coch
2,¿Puedo pedir hora para llevar el vehículo al t...,pued ped hor llev vehicul tall
3,¿Cómo va la reparación de mi coche?,com va repar coch
4,¿En qué fase está mi vehículo?,fas vehicul
5,Quiero saber si ya han arreglado mi coche,quier sab si arregl coch
6,¿Dónde descargo la factura?,dond descarg factur
7,Necesito la factura de la reparación,necesit factur repar
8,¿Cómo consigo el PDF de la factura?,com consig pdf factur
9,¿Por qué mi reparación está parada?,repar par


---

## 4. Vectorización

Para comparar frases hay que convertirlas en números. Vemos las dos representaciones clásicas:

### A. Bolsa de palabras (CountVectorizer)
Cada columna es una raíz del vocabulario y cada celda cuenta cuántas veces aparece en la pregunta.

### B. TF-IDF
Pondera cada palabra: baja el peso de las que aparecen en muchas preguntas (como *coch*) y sube el de las que identifican un tema (como *factur*).

In [4]:
# 4.1 Bolsa de palabras
vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(df['pregunta_limpia'])

print("--- Bolsa de palabras ---")
print("Vocabulario:", vectorizer_bow.get_feature_names_out())
print("Forma de la matriz:", X_bow.shape, "(preguntas x palabras)")

--- Bolsa de palabras ---
Vocabulario: ['actualiz' 'anterior' 'arregl' 'calcul' 'cambi' 'cit' 'coch' 'com'
 'consig' 'corre' 'cuand' 'cuant' 'cuent' 'cuest' 'descarg' 'dond'
 'electron' 'email' 'esper' 'factur' 'fas' 'hic' 'historial' 'hor' 'ir'
 'llev' 'man' 'modific' 'necesit' 'obra' 'par' 'pdf' 'ped' 'pendient'
 'pid' 'piez' 'preci' 'pued' 'quier' 'recog' 'repar' 'reserv' 'sab' 'si'
 'signif' 'tall' 'trabaj' 'ultim' 'va' 'vehicul' 'ver' 'vez']
Forma de la matriz: (24, 52) (preguntas x palabras)


In [5]:
# 4.2 TF-IDF
vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(df['pregunta_limpia'])

# Pesos de las palabras de la primera pregunta de "Factura"
fila = df.index[df['tema'] == 'Factura'][0]
pesos = pd.Series(X_tfidf[fila].toarray()[0], index=vectorizer_tfidf.get_feature_names_out())
print("Pregunta:", df.loc[fila, 'pregunta'])
print(pesos[pesos > 0].round(2).sort_values(ascending=False))

Pregunta: ¿Dónde descargo la factura?
descarg    0.61
dond       0.61
factur     0.49
dtype: float64


---

## 5. El modelo: buscar la pregunta más parecida

En vez de entrenar un clasificador, el asistente **compara** la pregunta del cliente con todas las que conoce usando la **similitud del coseno**: 1 significa que son iguales y 0 que no comparten ninguna palabra.

Si el mejor parecido no llega a un **umbral** (0,3), el asistente no se arriesga y pide que lo diga de otra forma. Así evitamos respuestas que no tienen nada que ver.

In [6]:
UMBRAL = 0.3

def responder(pregunta):
    # Limpiar y vectorizar con el mismo TF-IDF de las preguntas conocidas
    vector = vectorizer_tfidf.transform([preprocesar(pregunta)])

    # Parecido con cada pregunta conocida
    similitudes = cosine_similarity(vector, X_tfidf)[0]
    mejor = similitudes.argmax()

    if similitudes[mejor] < UMBRAL:
        return None, similitudes[mejor], "No te he entendido. Prueba a decirlo de otra forma o pregunta en Recepción."

    tema = df.loc[mejor, 'tema']
    return tema, similitudes[mejor], respuestas[tema]

tema, similitud, respuesta = responder("¿Dónde puedo bajar la factura?")
print("Tema:", tema, "· Similitud:", round(similitud, 2))
print("Respuesta:", respuesta)

Tema: Factura · Similitud: 0.68
Respuesta: Cuando la reparación está finalizada, entra en «Mis reparaciones» y descarga la factura en PDF.


---

## 6. Evaluación

Probamos el asistente con preguntas **nuevas**, escritas de otra forma, de las que sabemos qué tema deberían tener. Incluimos dos que el taller no contempla, para comprobar que el asistente reconoce cuándo no sabe la respuesta.

In [7]:
pruebas = [
    ("Me gustaría reservar hora para el lunes", 'Pedir cita'),
    ("¿Ya han terminado de arreglar mi vehículo?", 'Estado de la reparación'),
    ("No encuentro la factura de mi coche", 'Factura'),
    ("Me dicen que falta una pieza, ¿qué pasa?", 'Pendiente de pieza'),
    ("¿A qué hora paso a recoger el coche?", 'Recoger el coche'),
    ("¿Cuánto cuesta la reparación?", 'Precio'),
    ("Quiero cambiar mi correo", 'Cambiar email'),
    ("¿Puedo ver las reparaciones que le hicisteis antes?", 'Historial'),
    ("¿Abrís los domingos?", None),
    ("¿Vendéis neumáticos?", None),
    ("¿Hacéis cambio de aceite?", None),
]

resultados = []
for texto, esperado in pruebas:
    tema, similitud, _ = responder(texto)
    resultados.append({'pregunta': texto,
                       'esperado': esperado or 'Sin respuesta',
                       'obtenido': tema or 'Sin respuesta',
                       'similitud': round(similitud, 2),
                       'acierto': tema == esperado})

tabla = pd.DataFrame(resultados)
print("Aciertos:", tabla['acierto'].sum(), "de", len(tabla),
      f"({tabla['acierto'].mean() * 100:.0f} %)")
tabla

Aciertos: 10 de 11 (91 %)


,pregunta,esperado,obtenido,similitud,acierto
0,Me gustaría reservar hora para el lunes,Pedir cita,Pedir cita,0.46,True
1,¿Ya han terminado de arreglar mi vehículo?,Estado de la reparación,Estado de la reparación,0.40,True
2,No encuentro la factura de mi coche,Factura,Factura,0.46,True
3,"Me dicen que falta una pieza, ¿qué pasa?",Pendiente de pieza,Pendiente de pieza,0.61,True
4,¿A qué hora paso a recoger el coche?,Recoger el coche,Recoger el coche,0.52,True
5,¿Cuánto cuesta la reparación?,Precio,Precio,0.64,True
6,Quiero cambiar mi correo,Cambiar email,Cambiar email,0.48,True
7,¿Puedo ver las reparaciones que le hicisteis a...,Historial,Historial,0.66,True
8,¿Abrís los domingos?,Sin respuesta,Sin respuesta,0.00,True
9,¿Vendéis neumáticos?,Sin respuesta,Sin respuesta,0.00,True


**Análisis del fallo:** a *«¿Hacéis cambio de aceite?»* responde cómo cambiar el email, porque comparte la raíz *cambi* con *«¿Cómo cambio mi email?»*. El modelo compara palabras, no entiende su significado. Se corrige añadiendo frases a la base de conocimiento o subiendo el umbral, a costa de que deje de entender algunas preguntas válidas.

---

## 7. Chat con el asistente
Escribe tus preguntas como lo haría un cliente. Escribe **salir** para terminar.

In [ ]:
print("Asistente PitLineMotor: Hola, ¿en qué te puedo ayudar? (escribe 'salir' para terminar)")

while True:
    pregunta = input("Tú: ")
    if pregunta.strip().lower() == 'salir':
        print("Asistente: ¡Hasta luego!")
        break
    tema, similitud, respuesta = responder(pregunta)
    print("Asistente:", respuesta)
    print(f"   (tema: {tema or 'ninguno'} · similitud: {similitud:.2f})")